## Feature Engineering

This notebook focuses on reading the raw per-sample JSON records and the manifest to extract the three feature domains
1. URL features: lexical features
2. HTML features: structural features, ratio-based
3. JavaScript AST features: inline script parsing only; using Esprima, pre-order traversal to produce note-type sequence tokens.

In [1]:
# library imports
import json
import logging
import math
import os
import re
import sys
import esprima
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer, pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# directory mapping
RAW_DIR = "DA/raw"
PROCESSED_DIR = "DA/processed"
OUT_DIR = "DA/DA_features"
LOG_PATH = "logs/feature_engineering.log"

manifest_path = os.path.join(PROCESSED_DIR, "dataset_manifest.csv")
features_path = os.path.join(OUT_DIR, "features.csv")
js_seq_path = os.path.join(OUT_DIR, "js_node_sequences")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(js_seq_path, exist_ok=True)
os.makedirs("logs", exist_ok=True)

In [3]:
# logging configuration
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.flush = sys.stdout.flush
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_PATH), stream_handler],
    force=True,
)
log = logging.getLogger(__name__)

### Loading the dataset manifest

In [4]:
df = pd.read_csv(manifest_path)
log.info(
    f"Loaded manifest: {len(df)} rows "
    f"({(df["split"] == "train").sum()} train / {(df["split"] == "test").sum()} test)"
)
df.head()

2026-10-01 03:44:42,071 [INFO] Loaded manifest: 17269 rows (13815 train / 3454 test)


,sample_id,url,source,hostname,http_status,n_inline_scripts,n_external_scripts,html_length,label,split
0,ed2670faf3c8e66a,http://www.365k82.com/,phishing_db,www.365k82.com,200,1,3,66313,1,train
1,8b3a8e1ed1976a0c,https://jup.ag/,tranco,jup.ag,200,6,3,118284,0,train
2,0dbda29a0f82399e,https://cmail19.com/,tranco,cmail19.com,200,0,0,990,0,train
3,9e33e702e6648d96,https://smotrim.ru/,tranco,smotrim.ru,200,6,2,118290,0,train
4,1fc1eeff7dc9a1d2,https://www.javrate.com/,tranco,www.javrate.com,200,6,7,290622,0,train


### URL lexical features (F1-F12)

In [5]:
# IP address regular expressions
IP_RE = re.compile(r"^(?:\d{1,3}\.){3}\d{1,3}$")
IPV6_RE = re.compile(r"^\[?[0-9a-fA-F:]+\]?$")

In [6]:
def shannon_entropy(s):
    """Shannon entropy of a string, in bits. Empty string -> 0.0"""
    if not s:
        return 0.0
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    n = len(s)
    return -sum((c / n) * math.log2(c / n) for c in freq.values())

def is_ip_host(hostname):
    if not hostname:
        return False
    host = hostname.strip("[]")
    if IP_RE.match(host):
        return True
    if host.count(":") >= 2 and IPV6_RE.match(host):
        return True
    return False

def extract_url_features(record):
    url = record.get("seed_url") or ""
    parsed = urlparse(url if "://" in url else f"http://{url}")
    hostname = record.get("hostname") or parsed.hostname or ""
    labels = hostname.split(".") if hostname else []

    n = len(url)
    special_chars = sum(1 for ch in url if not ch.isalnum())

    return {
        "length": n,
        "dot_count": url.count("."),
        "hyphen_count": url.count("-"),
        "underscore_count": url.count("_"),
        "slash_count": url.count("/"),
        "digit_count": sum(1 for ch in url if ch.isdigit()),
        "has_ip_address": int(is_ip_host(hostname)),
        "has_https": int(parsed.scheme.lower() == "https"),
        "subdomain_count": max(0, len(labels) - 2),
        "query_length": len(parsed.query or ""),
        "entropy": round(shannon_entropy(url), 4),
        "special_char_ratio": round(special_chars / n, 6) if n else 0.0,
    }

### HTML structual features (F13-F27)

In [7]:
NULL_NREF_PATTERNS = {"#", "javascript:void(0)", "javascript:void(0);", "javascript:;"}
HIDDEN_STYLE_RE = re.compile(r"display\s*:\s*none|visibility\s*:\s*hidden", re.IGNORECASE)

In [8]:
def safe_ratio(n, d):
    return round(n / d, 6) if d else 0.0

def extract_html_features(html, hostname):
    if not html:
        return empty_html_features()
    try:
        soup = BeautifulSoup(html, "html.parser")
    except Exception as e:
        return empty_html_features()

    a_tags = soup.find_all("a")
    a_with_href = [a for a in a_tags if a.get("href")]
    a_no_href = [a for a in a_tags if not a.get("href")]
    link_href = [t for t in soup.find_all("link") if t.get("href")]
    img_src = [t for t in soup.find_all("img") if t.get("src")]
    script_src = [t for t in soup.find_all("script") if t.get("src")]

    # F20
    total_hyperlinks = len(a_tags) + len(link_href) + len(img_src) + len(script_src)

    empty_hrefs = sum(
        1 for a in a_with_href
        if a["href"].strip().lower() in NULL_NREF_PATTERNS
    )

    internal, external = 0, 0
    for a in a_with_href:
        href_host = urlparse(a["href"]).hostname
        if not href_host:
            internal += 1
        elif hostname and href_host.lower().removeprefix("www.") == hostname.lower().removeprefix("www."):
            internal += 1
        else:
            external += 1

    # F25
    title_tag = soup.find("title")
    missing_title = int(title_tag is None or not title_tag.get_text(strip=True))

    hidden_count = 0
    for tag in soup.find_all(style=True):
        if HIDDEN_STYLE_RE.search(tag["style"]):
            hidden_count += 1

    for tag in soup.find_all(["script", "style"]):
        tag.decompose()
    visible_text_len = len(re.sub(r"\s+", " ", soup.get_text(separator=" ")).strip())
    all_tags_count = len(soup.find_all(True))

    return{
        "script_link_ratio": safe_ratio(len(script_src), total_hyperlinks),
        # F13
        "css_link_ratio": safe_ratio(len(link_href), total_hyperlinks),
        # F14
        "image_link_ratio": safe_ratio(len(img_src), total_hyperlinks),
        # F15
        "anchor_link_ratio": safe_ratio(len(a_with_href), total_hyperlinks),
        # F16
        "empty_link_ratio": safe_ratio(empty_hrefs, total_hyperlinks),
        # F17
        "anchor_without_href_ratio": safe_ratio(len(a_no_href), total_hyperlinks),
        # F19
        "total_hyperlinks": total_hyperlinks,
        # F20
        "internal_link_ratio": safe_ratio(internal, total_hyperlinks),
        # F21
        "external_link_ratio": safe_ratio(external, total_hyperlinks),
        # F22
        "external_internal_ratio": safe_ratio(external, internal),
        # F23
        "total_forms": len(soup.find_all("form")),
        # F24
        "missing_title": missing_title,
        # F25
        "hidden_elements_count": hidden_count,
        # F26
        "text_to_tag_ratio": safe_ratio(visible_text_len, all_tags_count),
        }

def empty_html_features():
    return {
        "script_link_ratio": 0.0, "css_link_ratio": 0.0, "image_link_ratio": 0.0,
        "anchor_link_ratio": 0.0, "empty_link_ratio": 0.0, "anchor_without_href_ratio": 0.0, "total_hyperlinks": 0,
        "internal_link_ratio": 0.0, "external_link_ratio": 0.0,
        "external_internal_ratio": 0.0, "total_forms": 0, "missing_title": 1,
        "hidden_elements_count": 0, "text_to_tag_ratio": 0.0,
    }

### JavaScript AST node-type sequences

In [9]:
def traverse_ast(node, out):
    """Pre-order traversal grammatical 'type' tokens"""
    if isinstance(node, dict):
        t = node.get("type")
        if t:
            out.append(t)
        for k, v in node.items():
            if k == "type":
                continue
            traverse_ast(v, out)
    elif isinstance(node, list):
        for item in node:
            traverse_ast(item, out)

def extract_js_node_sequence(inline_scripts):
    """Returns token sequence list, n_parsed, and n_failed
       for a sample's inline scripts"""
    inline_scripts = [s for s in (inline_scripts or []) if s and s.strip()]
    all_tokens = []
    n_parsed, n_failed = 0, 0

    for script in inline_scripts:
        tree_dict = None
        try:
            tree_dict = esprima.parseScript(script, tolerant=True).toDict()
        except Exception:
            try:
                tree_dict = esprima.parseModule(script, tolerant=True).toDict()
            except Exception:
                n_failed += 1
                continue
        tokens = []
        traverse_ast(tree_dict, tokens)
        all_tokens.extend(tokens)
        n_parsed += 1

    return all_tokens, n_parsed, n_failed

### Extract features

In [10]:
import warnings
warnings.filterwarnings(action='ignore')

progress_every = 2000  # report progress after every 2000 processed samples
rows, n_missing_json, n_corrupt = [], 0, 0

for i, rec_row in enumerate(df.itertuples(), start=1):
    json_path = os.path.join(RAW_DIR, f"{rec_row.sample_id}.json")

    if not os.path.exists(json_path):
        log.warning(f"[missing] no raw json for sample_id={rec_row.sample_id} --- Dropped")
        n_missing_json += 1
        continue

    try:
        with open (json_path, "r", encoding="utf-8") as f:
            record = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError, OSError) as e:
        log.warning(f"[corrupt] {json_path} could not parse -> {e}")
        n_corrupt += 1
        continue

    url_feats = extract_url_features(record)
    html_feats = extract_html_features(record.get("raw_html"), record.get("hostname"))
    js_tokens, n_parsed, n_failed = extract_js_node_sequence(record.get("inline_scripts"))

    row = {
        "sample_id": rec_row.sample_id,
        "label": rec_row.label,
        "split": rec_row.split,
        "source": rec_row.source,
        "hostname": rec_row.hostname,
    }

    row.update({f"url_{k}": v for k, v in url_feats.items()})
    row.update({f"html_{k}": v for k, v in html_feats.items()})
    row["js_n_inline_scripts"] = len(record.get("inline_scripts") or [])
    row["js_n_parsed_scripts"] = n_parsed
    row["js_n_failed_scripts"] = n_failed
    row["js_node_sequence_length"] = len(js_tokens)
    rows.append(row)

    seq_path = os.path.join(js_seq_path, f"{rec_row.sample_id}.txt")
    with open (seq_path, "w", encoding="utf-8") as f:
        f.write(" ".join(js_tokens))

    if progress_every and i % progress_every == 0:
        log.info(f"Processed {i} / {len(df)} manifest rows")

log.info(
    f"Featurised {len(rows)} samples "
    f"({n_missing_json} mising raw JSON, {n_corrupt} corrupt discarded)"
)

2026-10-01 03:45:58,290 [INFO] Processed 2000 / 17269 manifest rows
2026-10-01 03:47:25,694 [INFO] Processed 4000 / 17269 manifest rows
2026-10-01 03:48:44,559 [INFO] Processed 6000 / 17269 manifest rows
2026-10-01 03:49:59,257 [INFO] Processed 8000 / 17269 manifest rows
2026-10-01 03:51:32,731 [INFO] Processed 10000 / 17269 manifest rows
2026-10-01 03:52:58,985 [INFO] Processed 12000 / 17269 manifest rows
2026-10-01 03:54:29,400 [INFO] Processed 14000 / 17269 manifest rows
2026-10-01 03:55:51,851 [INFO] Processed 16000 / 17269 manifest rows
2026-10-01 03:56:40,969 [INFO] Featurised 17269 samples (0 mising raw JSON, 0 corrupt discarded)


In [11]:
features_df = pd.DataFrame(rows)
features_df.to_csv(features_path, index=False)
log.info(f"Wrote engineered features -> {OUT_DIR} ({len(features_df)} rows)")
log.info(f"JS AST node-type sequences (pre-BPE) written to {js_seq_path} / for TF-IDF fitting at model time.")
features_df.head(10)

2026-10-01 03:56:41,120 [INFO] Wrote engineered features -> features (17269 rows)
2026-10-01 03:56:41,120 [INFO] JS AST node-type sequences (pre-BPE) written to features/js_node_sequences / for TF-IDF fitting at model time.


,sample_id,label,split,source,hostname,url_length,url_dot_count,url_hyphen_count,url_underscore_count,url_slash_count,...,html_external_link_ratio,html_external_internal_ratio,html_total_forms,html_missing_title,html_hidden_elements_count,html_text_to_tag_ratio,js_n_inline_scripts,js_n_parsed_scripts,js_n_failed_scripts,js_node_sequence_length
0,ed2670faf3c8e66a,1,train,phishing_db,www.365k82.com,17,1,0,0,2,...,0.060606,0.134615,1,0,0,2.835681,1,0,1,0
1,8b3a8e1ed1976a0c,0,train,tranco,jup.ag,14,1,0,0,2,...,0.059701,0.190476,1,0,0,2.110708,6,3,3,1836
2,0dbda29a0f82399e,0,train,tranco,cmail19.com,19,1,0,0,2,...,0.000000,0.000000,0,0,0,12.400000,0,0,0,0
3,9e33e702e6648d96,0,train,tranco,smotrim.ru,18,1,0,0,2,...,0.079051,0.317460,0,0,0,4.593583,6,4,2,4704
4,1fc1eeff7dc9a1d2,0,train,tranco,www.javrate.com,19,1,0,0,2,...,0.031250,0.045627,0,0,1,2.837321,6,5,1,586
5,d016e4e6a0127da8,1,train,phishtank,wallet-exoduss-io.wasmer.app,36,2,2,0,2,...,0.000000,0.000000,0,0,0,74.950820,0,0,0,0
6,d5db0e11daa6d171,1,train,phishtank,email-wp.weebly.com,28,2,1,0,3,...,0.035714,1.000000,1,0,5,2.138889,15,15,0,1229
7,770faf46f5efd74a,0,train,tranco,www.shangri-la.com,22,1,1,0,2,...,0.075472,0.195122,0,0,0,7.507282,4,3,1,272
8,e2396e3498e1fc18,0,train,tranco,freedom.to,18,1,0,0,2,...,0.183962,0.780000,0,0,1,13.344877,15,11,4,2476
9,69048adbdf415dbe,0,train,tranco,www.bsigroup.com,20,1,0,0,2,...,0.008949,0.010178,2,0,8,13.703178,35,30,5,10763


### JavaScript AST BPE/TF-IDF features

In [12]:
def load_js_sequence(sample):
    """
    Read a sample's flattened AST node-type sequence.
    Returns None if the file is missing, "" if it exists but is empty,
    otherwise the stripped sequence content.
    """
    seq_dir = os.path.join(js_seq_path, f"{sample}.txt")
    if not os.path.exists(seq_dir):
        return None
    with open(seq_dir, "r", encoding="utf-8") as f:
        content = f.read().strip()
    return content if content else ""

df["js_sequence"] = df["sample_id"].apply(load_js_sequence)

missing = df["js_sequence"].isna().sum()
empty = (df["js_sequence"] == "").sum()
log.info(f"Loaded JS node-type sequences for {len(df)} samples. "
         f"({missing} missing files, {empty} empty/failed-parse)")

# Preserve presence/absence as a feature for the full dataset
df["has_js_sequence"] = df["js_sequence"].apply(
    lambda x: isinstance(x, str) and x != ""
)

# Separate view for sequence-based modeling — only non-empty sequences
df_seq = df[df["has_js_sequence"]].reset_index(drop=True)

log.info(f"df: {len(df)} rows (full, with has_js_sequence flag) | "
         f"df_seq: {len(df_seq)} rows (non-empty sequences only)")

df.head()

2026-10-01 03:56:43,251 [INFO] Loaded JS node-type sequences for 17269 samples. (0 missing files, 2693 empty/failed-parse)
2026-10-01 03:56:43,254 [INFO] df: 17269 rows (full, with has_js_sequence flag) | df_seq: 14576 rows (non-empty sequences only)


,sample_id,url,source,hostname,http_status,n_inline_scripts,n_external_scripts,html_length,label,split,js_sequence,has_js_sequence
0,ed2670faf3c8e66a,http://www.365k82.com/,phishing_db,www.365k82.com,200,1,3,66313,1,train,,False
1,8b3a8e1ed1976a0c,https://jup.ag/,tranco,jup.ag,200,6,3,118284,0,train,Program ExpressionStatement CallExpression Mem...,True
2,0dbda29a0f82399e,https://cmail19.com/,tranco,cmail19.com,200,0,0,990,0,train,,False
3,9e33e702e6648d96,https://smotrim.ru/,tranco,smotrim.ru,200,6,2,118290,0,train,Program ExpressionStatement CallExpression Fun...,True
4,1fc1eeff7dc9a1d2,https://www.javrate.com/,tranco,www.javrate.com,200,6,7,290622,0,train,Program ExpressionStatement AssignmentExpressi...,True


#### JS presence vs. label
Quick sanity check before treating `has_js_sequence` as a feature, making sure it is not a collection artefact before reading anything causal into it

In [13]:
js_presence_by_label = pd.crosstab(df["has_js_sequence"], df["label"], normalize=True)
log.info("has_js_sequence vs. label (row-normalized):\n"
         f"{js_presence_by_label}")
js_presence_by_label

2026-10-01 03:56:43,266 [INFO] has_js_sequence vs. label (row-normalized):
label                   0         1
has_js_sequence                    
False            0.065725  0.090219
True             0.599050  0.245006


label,0,1
has_js_sequence,,
False,0.065725,0.090219
True,0.599050,0.245006


In [14]:
js_seq = df_seq['js_sequence'].tolist()
corpus = set()

for seq in js_seq:
    split_seq = seq.split()
    for node in split_seq:
        corpus.add(node)

log.info(f"[info] Number Unique nodes in parsed sequences: {len(corpus)} ==== "
         f"This also informs the vocabulary size in terms of BPE hyperparameter tuning.")

2026-10-01 03:56:45,991 [INFO] [info] Number Unique nodes in parsed sequences: 66 ==== This also informs the vocabulary size in terms of BPE hyperparameter tuning.


### Data splits

In [15]:
train_mask = df_seq["split"] == "train"
test_mask = df_seq["split"] == "test"

js_train = df_seq.loc[train_mask, "js_sequence"].to_numpy()
js_test = df_seq.loc[test_mask, "js_sequence"].to_numpy()
y_train = df_seq.loc[train_mask, "label"].to_numpy()
y_test = df_seq.loc[test_mask, "label"].to_numpy()

### AST-JS: BPE tokenisation and TF-IDF vectorisation

In [16]:
def script_sequences(doc):
    nodes = str(doc or "").split()
    sequences = []
    sequence = []

    for node in nodes:
        if node == "Program" and sequence:
            sequences.append(sequence)
            sequence = []
        sequence.append(node)   

    if sequence:
        sequences.append(sequence)
    return sequences

def calculate_compression_rate(docs, bpe_corpus):
    """Returns the mean rate, excluding empty sequences"""
    docs, bpe_corpus = list(docs), list(bpe_corpus)
    if len(docs) != len(bpe_corpus):
        raise ValueError("Original and encoded corpus lengths differ")
    rates = []
    for doc, bpe_doc in zip(docs, bpe_corpus):
        orig_length = len((str(doc) or "").split())
        bpe_length = len((str(bpe_doc) or "").split())
        if orig_length == 0:
            continue
        rates.append(100 * (orig_length - bpe_length) / orig_length)
    return float(np.mean(rates)) if rates else np.nan

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# wrap helper methods as a pipeline step
class ASTBPETokenizer(BaseEstimator, TransformerMixin):
    def __init__(self, vocab_size=100, min_frequency=1):
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        docs = list(X)
        nodes = sorted({node for doc in docs for node in str(doc or "").split()})
        self.node_to_symbol = {node: chr(0xE000 + i) for i, node in enumerate(nodes)}
        self.symbol_to_node = {symbol: node for node, symbol in self.node_to_symbol.items()}
        self.unk_symbol_ = chr(0xE000 + len(nodes))

        self.tokenizer_ = Tokenizer(BPE(unk_token="[UNK]"))
        trainer = BpeTrainer(vocab_size=self.vocab_size,
                            min_frequency=self.min_frequency,
                            special_tokens=["[UNK]"],
                            initial_alphabet=list(self.symbol_to_node))
        
        corpus = (self.map_nodes(nodes) for doc in docs for nodes in script_sequences(doc)) 
        self.tokenizer_.train_from_iterator(corpus, trainer=trainer)
        self.vocab_ = self.tokenizer_.get_vocab()
        self.multi_node_tokens = [token for token in self.vocab_
                                  if token != "[UNK]" and len(token) > 1]
        return self

    def map_nodes(self, nodes):
        return "".join(self.node_to_symbol.get(node, self.unk_symbol_) for node in nodes)

    def encode(self, doc):
        ids, tokens = [], []
        for nodes in script_sequences(doc):
            encoding = self.tokenizer_.encode(self.map_nodes(nodes), add_special_tokens=True)
            ids.extend(encoding.ids)
            tokens.extend(encoding.tokens)
        return ids, tokens

    def transform(self, X):
        return [" ".join(f"bpe_{i}" for i in self.encode(doc)[0]) for doc in list(X)]

    def recover_nodes(self, doc):
        """Recover known nodes. [UNK] stands for each unseen node type"""
        recovered = []
        for token in self.encode(doc)[1]:
            if token == "[UNK]":
                recovered.append("[UNK]")
            else:
                recovered.append(self.symbol_to_node[symbol] for symbol in token)
        return recovered

### BPE + TF-IDF hyperparameter search

In [18]:
BPE_GRID = {
    "vocab_size": [100, 250, 500, 1000],
    "min_frequency": [1, 2, 5, 10]
}

TFIDF_GRID = {
    "ngram_range": [(1, 1), (1, 2)],
    "min_df": [1, 2, 3],
    "sublinear_tf": [False, True],
    "max_features": [None, 250, 500]
}

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

pipe = Pipeline([
    ("bpe", ASTBPETokenizer()),
    ("tfidf", TfidfVectorizer(token_pattern=r"\S+", lowercase=False)),
    ("clf", DecisionTreeClassifier(random_state=42)),
    ])

n_bpe = len(BPE_GRID["vocab_size"]) * len(BPE_GRID["min_frequency"])
n_tfidf = (len(TFIDF_GRID["ngram_range"]) * len(TFIDF_GRID["min_df"])
           * len(TFIDF_GRID["sublinear_tf"]) * len(TFIDF_GRID["max_features"]))

log.info(f"Grid: {n_bpe} BPE configs x {n_tfidf} TF-IDF configs = {n_bpe * n_tfidf} total combos")

2026-10-01 03:56:46,220 [INFO] Grid: 16 BPE configs x 36 TF-IDF configs = 576 total combos


In [20]:
# subsample for the search phase only
SEARCH_SAMPLE_FRAC = 0.20

search_idx, _ = train_test_split(np.arange(len(js_train)),
                                 train_size=SEARCH_SAMPLE_FRAC,
                                 stratify=y_train,
                                 random_state=42)

js_search_docs = js_train[search_idx]
y_search = y_train[search_idx]

# holdout split for scoring configs
search_train_idx, search_val_idx = train_test_split(
    np.arange(len(js_search_docs)),
    test_size=0.2,
    stratify=y_search,
    random_state=42
)

log.info(f"Search subsample: {len(js_search_docs)}/{len(js_train)} samples "
      f"({SEARCH_SAMPLE_FRAC*100:.0f}%) | holdout val: {len(search_val_idx)} samples")

2026-10-01 03:56:46,229 [INFO] Search subsample: 2334/11671 samples (20%) | holdout val: 467 samples


In [21]:
import time
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer, matthews_corrcoef

# search_cv = [(search_train_idx, search_val_idx)]
cv_stratified_folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_grid = {
    "bpe__vocab_size": BPE_GRID["vocab_size"],
    "bpe__min_frequency": BPE_GRID["min_frequency"],
    "tfidf__ngram_range": TFIDF_GRID["ngram_range"],
    "tfidf__min_df": TFIDF_GRID["min_df"],
    "tfidf__sublinear_tf": TFIDF_GRID["sublinear_tf"],
    "tfidf__max_features": TFIDF_GRID["max_features"],
}

scoring = {
    "f1_macro": "f1_macro",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "mcc": "matthews_corrcoef"
}

search = RandomizedSearchCV(
    pipe, param_grid,
    n_iter=15,
    scoring=scoring,
    cv=cv_stratified_folds,
    refit='f1_macro',
    n_jobs=-1,
    verbose=3,
    error_score="raise"
)

t0 = time.time()
search.fit(js_search_docs, y_search)
log.info(f"Search finished in {(time.time() - t0)/60:.1f} minutes")

best_js_config = search.best_params_
best_js_score = search.best_score_
log.info(f"Best JS vectoriser config: {best_js_config} (CV F1-macro {best_js_score:.4f})")

Fitting 3 folds for each of 15 candidates, totalling 45 fits























[CV 2/3] END bpe__min_frequency=10, bpe__vocab_size=100, tfidf__max_features=500, tfidf__min_df=3, tfidf__ngram_range=(1, 1), tfidf__sublinear_tf=True; f1_macro: (test=0.780) mcc: (test=0.561) precision: (test=0.675) recall: (test=0.708) roc_auc: (test=0.777) total time= 2.4min
[CV 2/3] END bpe__min_frequency=10, bpe__vocab_size=100, tfidf__max_features=250, tfidf__min_df=2, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True; f1_macro: (test=0.783) mcc: (test=0.567) precision: (test=0.679) recall: (test=0.712) roc_auc: (test=0.781) total time= 2.5min




[CV 2/3] END bpe__min_frequency=5, bpe__vocab_size=500, tfidf__max_features=500, tfidf__min_df=2, tfidf__ngram_range=(1, 2), tfidf__sublinear_tf=True; f1_macro: (test=0.774) mcc: (test=0.547) precision: (test=0.674) recall: (test=0.686) roc_auc: (test=0.772) total time= 2.7min







[CV 2/3] END bpe__min_frequency=5, bpe__vocab_size=100, tfidf__max_fe

#### Fitting BPE/TF-IDF vectoriser with best hyperparameters

In [22]:
config = {k.split("__")[-1]: v for k, v in best_js_config.items()}
config

{'sublinear_tf': True,
 'ngram_range': (1, 1),
 'min_df': 1,
 'max_features': 500,
 'vocab_size': 100,
 'min_frequency': 1}

In [23]:
config = {k.split("__")[-1]: v for k, v in best_js_config.items()}

js_feature_pipe = Pipeline([('bpe', ASTBPETokenizer(config['vocab_size'], config['min_frequency'])),
                            ('tfidf', TfidfVectorizer(max_features=config['max_features'], 
                                                     min_df=config['min_df'],
                                                     ngram_range=config['ngram_range'], 
                                                     sublinear_tf=config['sublinear_tf'],
                                                     token_pattern=r"\S+", lowercase=False
                                                    ))])         
js_feature_pipe.fit(js_train.tolist())

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('bpe', ...), ('tfidf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,vocab_size,100
,min_frequency,1
Name,Type,Value
tokenizer_,Tokenizer,"Tokenizer(ver..."", """")]))"
unk_symbol_,str,'\ue042'
vocab_,dict,"{'[UNK]': 0, '\ue000': 1, '\ue001': 2, '\ue002': 3, ...}"
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",False


#### Reporting mean BPE compression rate when applied on training set.

In [25]:
X_train_bpe = js_feature_pipe['bpe'].transform(js_train.tolist())
compression_rate = calculate_compression_rate(js_train.tolist(), X_train_bpe)
log.info(f"[INFO] Mean compression rate: {compression_rate:.2f}%")

2026-10-01 04:36:08,870 [INFO] [INFO] Mean compression rate: 51.74%


### Final vectorization and domain unification

In [76]:
features_df

,sample_id,label,split,source,hostname,url_length,url_dot_count,url_hyphen_count,url_underscore_count,url_slash_count,...,html_external_link_ratio,html_external_internal_ratio,html_total_forms,html_missing_title,html_hidden_elements_count,html_text_to_tag_ratio,js_n_inline_scripts,js_n_parsed_scripts,js_n_failed_scripts,js_node_sequence_length
0,ed2670faf3c8e66a,1,train,phishing_db,www.365k82.com,17,1,0,0,2,...,0.060606,0.134615,1,0,0,2.835681,1,0,1,0
1,8b3a8e1ed1976a0c,0,train,tranco,jup.ag,14,1,0,0,2,...,0.059701,0.190476,1,0,0,2.110708,6,3,3,1836
2,0dbda29a0f82399e,0,train,tranco,cmail19.com,19,1,0,0,2,...,0.000000,0.000000,0,0,0,12.400000,0,0,0,0
3,9e33e702e6648d96,0,train,tranco,smotrim.ru,18,1,0,0,2,...,0.079051,0.317460,0,0,0,4.593583,6,4,2,4704
4,1fc1eeff7dc9a1d2,0,train,tranco,www.javrate.com,19,1,0,0,2,...,0.031250,0.045627,0,0,1,2.837321,6,5,1,586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17264,6ce2596cc694c260,0,test,tranco,www.branch.io,17,1,0,0,2,...,0.053398,0.110000,1,0,3,6.891626,13,9,4,6878
17265,6192124d07017564,0,test,tranco,test-ipv6.com,21,1,1,0,2,...,0.043478,0.076923,1,0,26,23.931298,4,4,0,301
17266,79faa0e263739751,0,test,tranco,www.ewg.org,15,1,0,0,2,...,0.048309,0.060241,2,0,1,6.853155,6,4,2,453
17267,c20822a5c8fe7de9,0,test,tranco,www.sante.fr,16,1,0,0,2,...,0.039474,0.081081,1,0,0,11.832727,1,0,1,0


In [77]:
train_full = df[df["split"] == "train"]

X_js_train = js_feature_pipe.transform(train_full["js_sequence"].tolist())
tfidf_names = [f"tfidf_{n.replace(' ', '__')}" 
               for n in js_feature_pipe["tfidf"].get_feature_names_out()]
 
tfidf_vector = pd.DataFrame(X_js_train.toarray() ,columns=tfidf_names, index=train_full.index)
tfidf_vector['split'] = 'train'

In [81]:
test_full = df[df["split"] == "test"]

X_js_train = js_feature_pipe.transform(test_full["js_sequence"].tolist())
tfidf_names = [f"tfidf_{n.replace(' ', '__')}" 
               for n in js_feature_pipe["tfidf"].get_feature_names_out()]
 
tfidf_vector_test = pd.DataFrame(X_js_train.toarray() ,columns=tfidf_names, index=test_full.index)
tfidf_vector_test['split'] = 'test'

In [82]:
full_js_df = pd.concat([tfidf_vector, tfidf_vector_test]).drop(columns='split')
full_js_df

,tfidf_bpe_1,tfidf_bpe_10,tfidf_bpe_11,tfidf_bpe_12,tfidf_bpe_13,tfidf_bpe_14,tfidf_bpe_15,tfidf_bpe_16,tfidf_bpe_17,tfidf_bpe_18,...,tfidf_bpe_90,tfidf_bpe_91,tfidf_bpe_92,tfidf_bpe_93,tfidf_bpe_94,tfidf_bpe_95,tfidf_bpe_96,tfidf_bpe_97,tfidf_bpe_98,tfidf_bpe_99
0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.172178,0.117506,0.098865,0.0,0.0,0.0,0.096522,0.0,0.0,0.000000,...,0.089619,0.297439,0.037569,0.000000,0.039904,0.061512,0.081229,0.254057,0.000000,0.126193
2,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.212749,0.036117,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.096305,0.210693,0.000000,0.361236,0.066395,0.035702,0.038037,0.056307,0.042792,0.215766
4,0.110441,0.121334,0.000000,0.0,0.0,0.0,0.063630,0.0,0.0,0.000000,...,0.123986,0.245164,0.135628,0.094492,0.144058,0.119939,0.164901,0.079270,0.102001,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17264,0.098489,0.100411,0.109238,0.0,0.0,0.0,0.144077,0.0,0.0,0.000000,...,0.127553,0.088281,0.119125,0.000000,0.111526,0.096778,0.100216,0.160138,0.150250,0.431819
17265,0.066403,0.108628,0.243239,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.193134,0.000000,0.193202,0.000000,0.000000,0.133094,0.067568,0.000000,0.000000,0.000000
17266,0.163850,0.144771,0.000000,0.0,0.0,0.0,0.075922,0.0,0.0,0.000000,...,0.173072,0.000000,0.130147,0.190892,0.183895,0.143107,0.152467,0.094582,0.171528,0.000000
17267,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [85]:
unified_df = pd.concat([features_df, full_js_df], axis=1)

tail_cols = [ "split", "label"]
unified_df = unified_df[[c for c in unified_df.columns if c not in tail_cols] + tail_cols]

unified_df = pd.concat(
    [unified_df.drop(columns=["label", "split"]),
     unified_df[[ "split", "label",]]],
    axis=1,
)

unified_df

,sample_id,source,hostname,url_length,url_dot_count,url_hyphen_count,url_underscore_count,url_slash_count,url_digit_count,url_has_ip_address,...,tfidf_bpe_92,tfidf_bpe_93,tfidf_bpe_94,tfidf_bpe_95,tfidf_bpe_96,tfidf_bpe_97,tfidf_bpe_98,tfidf_bpe_99,split,label
0,ed2670faf3c8e66a,phishing_db,www.365k82.com,17,1,0,0,2,5,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,train,1
1,8b3a8e1ed1976a0c,tranco,jup.ag,14,1,0,0,2,0,0,...,0.037569,0.000000,0.039904,0.061512,0.081229,0.254057,0.000000,0.126193,train,0
2,0dbda29a0f82399e,tranco,cmail19.com,19,1,0,0,2,2,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,train,0
3,9e33e702e6648d96,tranco,smotrim.ru,18,1,0,0,2,0,0,...,0.000000,0.361236,0.066395,0.035702,0.038037,0.056307,0.042792,0.215766,train,0
4,1fc1eeff7dc9a1d2,tranco,www.javrate.com,19,1,0,0,2,0,0,...,0.135628,0.094492,0.144058,0.119939,0.164901,0.079270,0.102001,0.000000,train,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17264,6ce2596cc694c260,tranco,www.branch.io,17,1,0,0,2,0,0,...,0.119125,0.000000,0.111526,0.096778,0.100216,0.160138,0.150250,0.431819,test,0
17265,6192124d07017564,tranco,test-ipv6.com,21,1,1,0,2,1,0,...,0.193202,0.000000,0.000000,0.133094,0.067568,0.000000,0.000000,0.000000,test,0
17266,79faa0e263739751,tranco,www.ewg.org,15,1,0,0,2,0,0,...,0.130147,0.190892,0.183895,0.143107,0.152467,0.094582,0.171528,0.000000,test,0
17267,c20822a5c8fe7de9,tranco,www.sante.fr,16,1,0,0,2,0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,test,0


### Save final csv for model training

In [86]:
unified_df.to_csv(features_path, index=False)
log.info(f"Saved feature set csv --> {features_path}")

2026-10-01 05:18:22,767 [INFO] Saved feature set csv --> features/features.csv
